In [3]:
!pip install shapely pyproj

In [4]:
"""
Wellington Street — Segment Validation for Counter 32493

Purpose:
Confirm which Wellington Street treatment segment is closest to cycling
counter 32493.

Wellington Street has three treatment segments with different intervention
timing:

    - 5881 / 5882: Painted -> Protected bike lane in 2015
    - 5883:        Painted -> Protected bike lane in 2019

The counter is matched to the nearest segment by measuring the GPS distance
from each counter point to the road geometry.

Input file:
    - ref_streets_spatial.xlsx
      Street segment geometry stored in the wkt_geom column

Counter coordinates:
    - Counter 32493 North-bound
    - Counter 32493 South-bound

Expected result:
    Both counter points are closest to segment 5881.

This code can run as either:
    - a Python script (.py), or
    - a Jupyter Notebook (.ipynb).
"""

from pathlib import Path

import pandas as pd
from pyproj import Transformer
from shapely import wkt
from shapely.geometry import Point


# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------

# Works in both Python scripts and Jupyter notebooks.
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

STREETS_SPATIAL_PATH = BASE_DIR / "ref_streets_spatial.xlsx"

WELLINGTON_SEGMENT_IDS = [5881, 5882, 5883]

COUNTER_POINTS = {
    "32493_North_bound": (-37.807171, 144.985903),
    "32493_South_bound": (-37.807664, 144.985967),
}

print("Working directory:", BASE_DIR)
print("Input file:", STREETS_SPATIAL_PATH)


# -------------------------------------------------------------------
# 1. Check input file
# -------------------------------------------------------------------

if not STREETS_SPATIAL_PATH.exists():
    raise FileNotFoundError(
        f"Input file not found: {STREETS_SPATIAL_PATH}\n"
        "Make sure ref_streets_spatial.xlsx is in the same folder "
        "as the script or notebook."
    )


# -------------------------------------------------------------------
# 2. Load Wellington Street segment geometry
# -------------------------------------------------------------------

streets = pd.read_excel(STREETS_SPATIAL_PATH)

required_columns = {
    "street_segment_id",
    "wkt_geom",
}

missing_columns = required_columns - set(streets.columns)

if missing_columns:
    raise KeyError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

wellington_segments = streets.loc[
    streets["street_segment_id"].isin(WELLINGTON_SEGMENT_IDS)
].copy()

if wellington_segments.empty:
    raise ValueError(
        "No Wellington Street segments were found for IDs "
        f"{WELLINGTON_SEGMENT_IDS}."
    )

wellington_segments["geom"] = (
    wellington_segments["wkt_geom"]
    .apply(wkt.loads)
)

print(
    "\nWellington Street segments:",
    sorted(
        wellington_segments["street_segment_id"]
        .tolist()
    ),
)


# -------------------------------------------------------------------
# 3. Set up coordinate transformation
# -------------------------------------------------------------------

# Counter coordinates:
# EPSG:7844 = GDA2020 geographic coordinates
#
# Street geometry:
# EPSG:7899 = GDA2020 / Vicgrid
#
# Transforming into the same projected CRS allows distance
# calculations in metres.

transformer = Transformer.from_crs(
    "EPSG:7844",
    "EPSG:7899",
    always_xy=True,
)


# -------------------------------------------------------------------
# 4. Calculate distance to each Wellington Street segment
# -------------------------------------------------------------------

results = []

for point_name, (latitude, longitude) in COUNTER_POINTS.items():

    x, y = transformer.transform(
        longitude,
        latitude,
    )

    counter_point = Point(x, y)

    for _, segment in wellington_segments.iterrows():

        distance_m = counter_point.distance(
            segment["geom"]
        )

        results.append({
            "counter_point": point_name,
            "segment_id": int(
                segment["street_segment_id"]
            ),
            "distance_m": round(
                distance_m,
                2,
            ),
        })


results_df = pd.DataFrame(results)


# -------------------------------------------------------------------
# 5. Create distance comparison table
# -------------------------------------------------------------------

distance_table = results_df.pivot(
    index="counter_point",
    columns="segment_id",
    values="distance_m",
)

print(
    "\nDistance from counter 32493 "
    "to each Wellington Street segment (metres):\n"
)

print(distance_table.to_string())


# -------------------------------------------------------------------
# 6. Identify the closest segment
# -------------------------------------------------------------------

closest_segments = distance_table.idxmin(axis=1)

print("\nClosest segment for each counter point:")

print(
    closest_segments.to_string()
)


# -------------------------------------------------------------------
# 7. Build simple result summary
# -------------------------------------------------------------------

summary_rows = []

for counter_point in distance_table.index:

    closest_segment = int(
        closest_segments.loc[counter_point]
    )

    closest_distance = float(
        distance_table.loc[
            counter_point,
            closest_segment,
        ]
    )

    summary_rows.append({
        "counter_point": counter_point,
        "closest_segment": closest_segment,
        "distance_m": closest_distance,
    })

summary_df = pd.DataFrame(summary_rows)

print("\nSummary:")
print(summary_df.to_string(index=False))


# -------------------------------------------------------------------
# 8. Validate expected result
# -------------------------------------------------------------------

all_match_5881 = (
    summary_df["closest_segment"] == 5881
).all()

if all_match_5881:
    print(
        "\nConfirmed: both counter points "
        "are closest to segment 5881."
    )
else:
    print(
        "\nWarning: not all counter points "
        "were matched to segment 5881."
    )


# -------------------------------------------------------------------
# Final interpretation
# -------------------------------------------------------------------

print(
    """
Analysis complete.

Result:
- Both directions of counter 32493 are expected to be approximately
  6–7 m from segment 5881.
- Segment 5882 is farther away.
- Segment 5883 is substantially farther away.
- Counter 32493 is therefore matched to segment 5881.

Intervention context:
- Segment 5881 changed from a painted bike lane to a protected bike lane
  in 2015.
- The later 2019 protected-bike-lane conversion relates to segment 5883,
  not segment 5881.
"""
)

Working directory: c:\Users\lqye9\OneDrive\Desktop\deakin\2026 T2\SIT378_SIT782 - Team Project B - Execution and Delivery\git\team_a\notebooks\jin_kim
Input file: c:\Users\lqye9\OneDrive\Desktop\deakin\2026 T2\SIT378_SIT782 - Team Project B - Execution and Delivery\git\team_a\notebooks\jin_kim\ref_streets_spatial.xlsx

Wellington Street segments: [5881, 5882, 5883]

Distance from counter 32493 to each Wellington Street segment (metres):

segment_id         5881    5882    5883
counter_point                          
32493_North_bound  6.87   67.34  309.08
32493_South_bound  6.01  120.62  362.53

Closest segment for each counter point:
counter_point
32493_North_bound    5881
32493_South_bound    5881

Summary:
    counter_point  closest_segment  distance_m
32493_North_bound             5881        6.87
32493_South_bound             5881        6.01

Confirmed: both counter points are closest to segment 5881.

Analysis complete.

Result:
- Both directions of counter 32493 are expected to